# 01.5 Building Models with `nn.Module` / 用 `nn.Module` 构建模型

这一节开始真正进入“模型”本身。  
This notebook is where we start working with actual models.

`nn.Module` 是 `PyTorch` 模型的核心抽象。  
`nn.Module` is the core abstraction behind `PyTorch` models.

重点概念 / Key concepts:

- 模块 / module
- 层 / layer
- 前向传播 / forward pass
- 参数 / parameters
- 子模块 / submodules
- 多层感知机 / multilayer perceptron, MLP

## 学习目标 / Learning Goals

学完后你应该能 / After this notebook, you should be able to:

1. 理解 `nn.Module` 的基本结构 / Understand the basic structure of `nn.Module`.
2. 自己定义简单模型类 / Define simple model classes yourself.
3. 区分层和参数 / Distinguish layers from parameters.
4. 看懂 `forward` 在做什么 / Read what `forward` is doing.
5. 构建一个简单 MLP / Build a simple MLP.
6. 为后续训练循环准备模型部分 / Prepare the model component for later training loops.

In [ ]:
import torch
import torch.nn as nn

## 1. `nn.Module` 的最小结构 / The Minimal Structure of `nn.Module`

一个最小模型通常有两部分：  
A minimal model usually has two parts:

1. `__init__`：定义层 / define layers
2. `forward`：定义数据如何流过这些层 / define how data flows through the layers

In [ ]:
class SimpleLinearModel(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x):
        return self.linear(x)


model = SimpleLinearModel(in_features=2, out_features=1)
print(model)

这里的意思非常直接 / The meaning here is very direct:

- `nn.Linear(2, 1)`：输入 2 个特征，输出 1 个值 / input 2 features, output 1 value
- `forward`：数据进来后，调用这层 / when data comes in, pass it through this layer

## 2. 前向传播 / Forward Pass

`forward` 决定了输入怎样变成输出。  
`forward` defines how inputs become outputs.

在 `PyTorch` 里，你通常不直接写 `model.forward(x)`，而是写 `model(x)`。  
In `PyTorch`, you usually do not write `model.forward(x)` directly. You write `model(x)`.

In [ ]:
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
out = model(x)

print("x.shape =", x.shape)
print("out =\n", out)
print("out.shape =", out.shape)

因为 `out_features=1`，所以输出 shape 是 `(batch_size, 1)`。  
Because `out_features=1`, the output shape is `(batch_size, 1)`.

In [ ]:
# 练习 1 / Exercise 1
# 创建一个线性层模型，输入维度 3，输出维度 2。
# Create a linear model with input dimension 3 and output dimension 2.
#
# 然后输入一个 shape 为 (4, 3) 的张量，并打印输出 shape。
# Then feed in a tensor of shape (4, 3) and print the output shape.

# model_ex =
# x_ex =
# out_ex =
# print(out_ex.shape)

In [ ]:
# 练习 1 参考答案 / Exercise 1 Reference Solution

model_ex = SimpleLinearModel(in_features=3, out_features=2)
x_ex = torch.randn(4, 3)
out_ex = model_ex(x_ex)
print(out_ex.shape)

## 3. 参数 / Parameters

模型之所以能学习，是因为里面有可更新的参数。  
A model can learn because it contains updateable parameters.

对 `nn.Linear` 来说，最典型的参数就是：  
For `nn.Linear`, the most typical parameters are:

- 权重 / weights
- 偏置 / bias

In [ ]:
for name, param in model.named_parameters():
    print(name)
    print("shape =", param.shape)
    print(param)
    print()

以后你看到 `model.parameters()` 或 `model.named_parameters()`，本质上就是在查看模型里能被学习的部分。  
When you later see `model.parameters()` or `model.named_parameters()`, you are essentially looking at the learnable parts of the model.

## 4. 加入激活函数 / Adding an Activation Function

只有线性层时，模型表达能力有限。  
With only linear layers, the model has limited expressive power.

所以常常在线性层之间加入非线性激活函数 / non-linear activation functions.  
That is why non-linear activation functions are often inserted between linear layers.

常见例子 / Common examples:

- `ReLU`
- `Sigmoid`
- `Tanh`

In [ ]:
class TwoLayerMLP(nn.Module):
    def __init__(self, in_features, hidden_features, out_features):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_features, out_features)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x


mlp = TwoLayerMLP(in_features=2, hidden_features=4, out_features=1)
print(mlp)

In [ ]:
x = torch.randn(5, 2)
out = mlp(x)
print("x.shape =", x.shape)
print("out.shape =", out.shape)
print(out)

这里的 shape 流动 / Shape flow here:

- 输入 / input: `(5, 2)`
- 第一层后 / after first layer: `(5, 4)`
- 第二层后 / after second layer: `(5, 1)`

理解 shape 流动非常关键。  
Understanding shape flow is extremely important.

In [ ]:
# 练习 2 / Exercise 2
# 定义一个模型 / Define a model:
# 输入维度 / input dim = 4
# 隐藏维度 / hidden dim = 8
# 输出维度 / output dim = 3
#
# 然后输入一个 shape 为 (6, 4) 的张量，并打印输出 shape。

# model2 =
# x2 =
# out2 =
# print(out2.shape)

In [ ]:
# 练习 2 参考答案 / Exercise 2 Reference Solution

model2 = TwoLayerMLP(in_features=4, hidden_features=8, out_features=3)
x2 = torch.randn(6, 4)
out2 = model2(x2)
print(out2.shape)

## 5. `nn.Sequential` / `nn.Sequential`

如果模型结构是简单串联的，可以用 `nn.Sequential` 写得更紧凑。  
If the model is a simple chain of layers, `nn.Sequential` can make the code more compact.

In [ ]:
seq_model = nn.Sequential(
    nn.Linear(2, 4),
    nn.ReLU(),
    nn.Linear(4, 1),
)

print(seq_model)

x = torch.randn(3, 2)
out = seq_model(x)
print("out.shape =", out.shape)

`nn.Sequential` 很适合简单结构，但当你需要分支、跳连、多个输入输出时，通常还是写自定义 `forward` 更清晰。  
`nn.Sequential` is great for simple chains, but when you need branches, skip connections, or multiple inputs/outputs, a custom `forward` is usually clearer.

## 6. 输出层和任务类型 / Output Layer and Task Type

模型最后一层怎么设计，和任务类型强相关。  
The design of the final layer depends strongly on the task type.

常见情况 / Common cases:

- 回归 / regression: 输出一个或多个连续值 / output one or more continuous values
- 二分类 / binary classification: 输出一个 logit 或概率 / output a logit or probability
- 多分类 / multiclass classification: 输出每一类的分数 / output one score per class

这一点在下一节讲损失函数时会更重要。  
This becomes even more important in the next notebook on loss functions.

In [ ]:
reg_model = TwoLayerMLP(in_features=2, hidden_features=4, out_features=1)
cls_model = TwoLayerMLP(in_features=2, hidden_features=4, out_features=3)

x = torch.randn(4, 2)
print("regression output shape =", reg_model(x).shape)
print("classification output shape =", cls_model(x).shape)

In [ ]:
# 练习 3 / Exercise 3
# 看下面三个任务，判断输出维度通常应该是多少。
# For each task below, decide what the output dimension is usually.
#
# 1. 预测房价 / predict house price
# 2. 判断邮件是否垃圾邮件 / spam or not spam
# 3. 把图片分成 10 类 / classify image into 10 classes
#
# 请用一句话写下你的判断。
# Write your answer in one sentence.

参考回答 / Reference answer:

- 房价回归 / house-price regression: 通常输出维度为 1 / usually output dimension 1
- 二分类 / binary classification: 常见输出维度为 1 或 2，取决于损失设计 / often output dimension 1 or 2, depending on the loss setup
- 10 分类 / 10-class classification: 通常输出维度为 10 / usually output dimension 10

## 7. 小结 / Summary

本节最重要的不是记住类名，而是理解模型定义的固定结构：  
The key lesson here is not memorizing class names, but understanding the fixed structure of model definition:

1. 在 `__init__` 中定义层 / define layers in `__init__`
2. 在 `forward` 中定义数据流 / define data flow in `forward`
3. 通过参数让模型可学习 / let the model learn through parameters

你现在应该能回答 / You should now be able to answer:

1. 为什么 `nn.Module` 是模型的核心抽象？ / Why is `nn.Module` the core abstraction for models?
2. 为什么层通常写在 `__init__`，而不是 `forward` 里？ / Why are layers usually defined in `__init__` instead of inside `forward`?
3. 输出维度为什么和任务类型有关？ / Why does output dimension depend on task type?
4. `nn.Sequential` 和自定义 `forward` 各适合什么场景？ / When is `nn.Sequential` suitable and when is a custom `forward` better?

下一步建议 / Suggested next step:

- 进入损失函数与优化器 notebook，把“模型输出”接到“如何学习”上 / Move to the loss-and-optimizer notebook to connect model outputs to learning.